# 策略概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

**MST PartialCorr Cointegration** 以**偏相關網路圖**作為配對候選生成器，取代 HDBSCAN 聚類或 GICS 產業分組——這是命題 1 的**第三種候選生成方式**（同 SSD-DTW 排序、同共整合篩選、同交易端，唯一變因為候選生成）：

1. 日報酬 → **因子殘差化**（移除市場＋產業共動）
2. **Ledoit-Wolf 收縮精確矩陣** → 標準化為**偏相關矩陣**
3. 以 $1-|\rho_{ij}^{part}|$ 為距離建**最小生成樹（MST）＋ kNN** → 候選邊集合
4. 逐候選邊：雙向 OLS + ADF／半衰期／Hurst／零穿越／成本可行性過濾
5. SSD／DTW 距離 → PCA 融合 PC1 排序，取前 `top_n`
6. 交易期以 `ignore_ols_alpha=True` 於標準化空間重建 spread

實作模組：`strategies/formation/MST_PartialCorr_Cointegration.py`（`partial_corr=True, graph_method="mst+knn", knn_k=5, factor_residual=True, method="ssd_dtw_pca"`）。


## 為何用「偏相關圖」作候選生成

普通相關會把「A、B 只因同時與市場／龍頭股 C 相關」的**間接連結**也算進來。**偏相關**（精確矩陣 $\Theta=\Sigma^{-1}$ 的標準化負值 $\rho_{ij}^{part}=-\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$）在**控制其餘全部股票**後，只保留 A–B 的**直接共動**——這正是共整合配對的經濟來源（Kenett et al. 2010；Mantegna 1999 的 MST 精神）。

形成窗 $T\approx252 < N\approx350$（樣本共變異矩陣退化、不可逆），故以 **Ledoit-Wolf 收縮**估計精確矩陣。MST 取最強骨幹的 $N-1$ 條邊、kNN 補局部強連結，把 $\binom{N}{2}\approx6$ 萬對候選降到**千級**，且聚焦於統計上最強的直接連結。

::: {.callout-important}

**實測為負面結果（可寫的消融證據）**：三方對照中 MST 圖候選最差（中位 Sharpe 顯著為負），且與獲利的聚類配對幾乎零重疊——顯示候選生成的關鍵**不是**稀疏優雅的圖拓撲，而是給強距離排序器一個「含得到真正可交易對」的豐富候選池。此結果反證「HDBSCAN 聚類的優勢是特定的、非任何候選生成器皆可」。詳見 `notebooks/comparison.ipynb` 與 `analysis/regime_cost_dsr_eval.py`。

:::


# 參考文獻與引用對應


## 文獻 1：Mantegna (1999)

> Mantegna, R. N. (1999). Hierarchical structure in financial markets. *The European Physical Journal B*, **11**(1), 193–197.

以相關距離 $d_{ij}=\sqrt{2(1-\rho_{ij})}$ 建**最小生成樹**揭露市場的階層結構——本策略「階段 3」的圖候選骨幹來源（本實作改以偏相關的 $1-|\rho^{part}|$ 為距離）。


## 文獻 2：Ledoit & Wolf (2004)

> Ledoit, O., & Wolf, M. (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, **88**(2), 365–411.

收縮估計量在 $T<N$ 時仍給出**良置（可逆）**的共變異／精確矩陣——本策略「階段 2」偏相關計算的基礎。


## 文獻 3：Kenett et al. (2010)

> Kenett, D. Y., Tumminello, M., Madi, A., Gur-Gershgoren, G., Mantegna, R. N., & Ben-Jacob, E. (2010). Dominating clasp of the financial sector revealed by partial correlation analysis of the stock market. *PLoS ONE*, **5**(12), e15032.

以**偏相關**移除市場中介的間接相關、揭露個股間的直接依存——本策略以偏相關（而非普通相關）建圖的動機依據。


## 文獻 4：Avellaneda & Lee (2010)／Engle & Granger (1987)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica*, **55**(2), 251–276.

前者為「階段 1」因子殘差化（以特殊性報酬建模）依據；後者為「階段 4」ADF 共整合檢定依據。


# 各階段行為


## 階段 1：因子殘差化（依據：Avellaneda & Lee 2010）

對形成窗日報酬矩陣 $R\in\mathbb{R}^{(T-1)\times N}$ 依序移除兩層因子：

1. **市場因子** $f^{mkt}_t=\frac{1}{N}\sum_i R_{t,i}$，逐股對 $[1,f^{mkt}]$ OLS 取殘差 $e^{(1)}$
2. **產業因子**（≥3 檔的產業）：$g^{s}_t=$ 同產業市場殘差橫斷面平均，逐股再取殘差 $e^{(2)}$

偏相關圖建於**特殊性報酬** $e^{(2)}$，避免被市場 β 齊漲齊跌主導、且更耐 regime。由 `_utils.py` 的 `_residualize_returns()` 執行（`factor_residual=True`）。


## 階段 2：偏相關矩陣（依據：Ledoit & Wolf 2004／Kenett et al. 2010）

對標準化殘差報酬 $R_s$ 以 Ledoit-Wolf 收縮估計精確矩陣 $\Theta$（$T<N$ 安全），標準化為偏相關：

$$\rho_{ij}^{part} = -\,\frac{\Theta_{ij}}{\sqrt{\Theta_{ii}\,\Theta_{jj}}}$$

（`partial_corr=False` 時退回普通相關矩陣作對照。）


## 階段 3：MST ＋ kNN 候選邊（依據：Mantegna 1999）

以邊權 $|\rho_{ij}^{part}|$、距離 $D_{ij}=1-|\rho_{ij}^{part}|$ 建圖：

- **MST**（`scipy.sparse.csgraph.minimum_spanning_tree`）：$N-1$ 條最強骨幹邊
- **kNN**：每檔股票連到 $|\rho^{part}|$ 最強的 `knn_k`（=5）個鄰居
- 候選邊 = 兩者聯集（`graph_method="mst+knn"`），並以 `pcorr_threshold` 過濾弱邊

候選集由 $\binom{N}{2}\approx6$ 萬對降到約 1–2 千條邊，只對這些邊做共整合檢定。


## 階段 4：逐候選邊共整合篩選（依據：Engle & Granger 1987）

對每條候選邊 $(a,b)$，複用 `_utils.py` 的核心 gate（與 HDBSCAN 系列一致）：

1. log-price 相關 $\ge$ `min_corr`（0.5）
2. 雙向 OLS + ADF，取 p 值較小方向；$p\le$ `adf_pvalue_threshold`（0.01）
3. OU 半衰期 $\in[1,60]$、Hurst $<$ 門檻、零穿越 $\ge$ `min_zero_crossings`
4. 成本可行性（`use_cost_filter`）：$2\sigma_{spread}\ge$ 往返成本 0.58%
5. （`use_fdr` 時）BH-FDR 多重檢定校正


## 階段 5：SSD-DTW-PCA 排序

對通過共整合的邊，於 Z-Score 標準化對數價格空間計算 SSD 與 Sakoe-Chiba DTW 距離，以 `StandardScaler` + PCA 融合取 **PC1** 升序排序（`method="ssd_dtw_pca"`），取前 `top_n`。此為 A 段實測證明主導最終選對的**強排序器**，與勝出的 HDBSCAN Cluster+Resid 路徑一致。


## 階段 6：交易期參數使用

`ignore_ols_alpha=True`，交易期於**標準化空間**重建 spread：

$$P^{\prime}_{i,t}=\frac{\ln P_{i,t}-\mu^{form}_{\ln P_i}}{\sigma^{form}_{\ln P_i}},\qquad \text{Spread}_t=P^{\prime}_{A,t}-\beta\,P^{\prime}_{B,t}$$

輸出 `Ticker_A/B, Rank, Hedge_Ratio, OLS_Alpha, Spread_Mean/Std, Log_Mean/Std_A/B` 供 `zscore_trading.py`（路徑 B）使用。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗 $F$ / 滾動步長 | 252 / 21 交易日 | — |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 每期配對數 |
| `factor_residual` | **True** | 建圖前移除市場＋產業因子 |
| `partial_corr` | **True** | 偏相關（Ledoit-Wolf 精確矩陣）vs 普通相關 |
| `graph_method` / `knn_k` | `mst+knn` / 5 | 候選邊生成方式 |
| `pcorr_threshold` | 0.0 | 候選邊 $|\rho^{part}|$ 下限 |
| `method` | `ssd_dtw_pca` | PC1 融合排序（強排序器） |
| `adf_pvalue_threshold` | 0.01 | ADF 顯著水準 |
| 半衰期 / Hurst / 零穿越 | $[1,60]$ 日 / 門檻 / $\ge$ 設定 | 均值回歸過濾 |
| `ignore_ols_alpha` | True | 標準化空間 spread（路徑 B） |
